## Table of Contents

* [1. Overview](#1-Overview)
* [2. Building Model](#2-building-model)
  * [2.1 Dimensions and Floors](#21-dimensions-and-floors)
  * [2.2 Floor Transitions](#22-floor-transitions)
  * [2.3 Two Exits](#23-two-exits)
* [3. Theory](#3-theory)
  * [3.1 Ramp Height Function](#31-ramp-height-function)
  * [3.2 Surface Height](#32-surface-height)
  * [3.3 Total Cost](#33-total-cost)
* [4. Implementation](#4-implementation)
* [5. Trajectory Analysis (2-D projections per floor)](#5-trajectory-analysis-2-d-projections-per-floor)
* [6. Exit Selection Analysis](#6-exit-selection-analysis)
* [7. 3-D Architectural Visualization and Animation](#7-3-d-architectural-visualization-and-animation)
* [8. Quantitative Analysis](#8-quantitative-analysis)
  * [8.1 Distance to Nearest Exit Over Time](#81-distance-to-nearest-exit-over-time)
* [9. Discussion](#9-discussion)
  * [Q1: How does the soft-min cost function enable natural exit selection?](#q1-how-does-the-soft-min-cost-function-enable-natural-exit-selection)
  * [Q2: How do the ramps enable floor-to-floor transitions without discrete state?](#q2-how-do-the-ramps-enable-floor-to-floor-transitions-without-discrete-state)
  * [Q3: What role does the smoothness term play?](#q3-what-role-does-the-smoothness-term-play)
  * [Q4: Parameter sensitivity](#q4-parameter-sensitivity)
* [10. References](#10-references)

---

## 1. Overview

This assignment extends the multi-particle cost-function framework into a full 3-D three-floor building evacuation simulation.  All motion emerges from minimising a scalar cost function — no collision detection, no path-planning algorithms, and no rule-based floor switching.  Vertical transitions are handled by two ramps modelled as geometric surface patches; the cost function penalises deviation from those surfaces so particles naturally descend to the ground floor and exit through one of two exits chosen via a soft-min formulation.

**Key design choices:**
* Ramp height follows the linear projection formula from the assignment specification.
* Surface-adherence cost uses a quadratic penalty weighted by `w_height`.
* Goal attraction uses the soft-min of two squared distances (temperature τ).
* Wall repulsion uses a log-barrier penalty.
* Agent repulsion uses a short-range quadratic repulsion.
* Smoothness term penalises large jumps between iterations.
* Gradient is approximated by central finite differences in all three dimensions.

---

## 2. Building Model

### 2.1 Dimensions and Floors

| Floor | z-height |
|-------|----------|
| Ground | 0.0 |
| First  | H = 4.0 |
| Second | 2H = 8.0 |

Each floor is a 12 × 10 rectangular room subdivided by interior walls to form a
sufficiently challenging layout (L-shaped corridors, a central dividing wall, and a
side alcove).  Back walls are taller to frame the scene; interior walls are lower so
agents remain visible from the camera.

### 2.2 Floor Transitions

Two ramps connect:
* **Ramp A** – second floor (z = 2H) → first floor (z = H)
* **Ramp B** – first floor  (z = H)  → ground floor (z = 0)
Each ramp is a capsule corridor with half-width `r_ramp = 0.9`.

### 2.3 Two Exits

* Exit 1 at the south-west corner of the ground floor
* Exit 2 at the south-east corner of the ground floor

Exit assignment is never hard-coded; the soft-min goal cost lets each agent
gravitate toward whichever exit is currently cheaper.



---

## 3. Theory

### 3.1 Ramp Height Function

For a ramp with centreline endpoints $A$, $B$ connecting floors at $z_0$ and $z_1$:

$$t = \frac{(P - A) \cdot v}{v \cdot v}, \quad v = B - A$$

$$u(x,y) = \operatorname{clip}(t, 0, 1)$$

$$z_\text{ramp}(x,y) = z_0 + (z_1 - z_0)\, u(x,y)$$

The ramp footprint is a capsule of half-width $r$:

$$\mathcal{R} = \{(x,y) \mid \operatorname{dist}((x,y), [A,B]) \le r\}$$

### 3.2 Surface Height

$$z_\text{surf}(x,y) = (1 - g)\, z_\text{floor} + g\, z_\text{ramp}(x,y)$$

where $g = 1$ inside the ramp footprint and $g = 0$ outside.

### 3.3 Total Cost

$$C = C_\text{goal} + C_\text{walls} + C_\text{height} + C_\text{smooth} + C_\text{repulsion}$$

**Goal (soft-min):**

$$C_\text{goal}(p) = -\tau \log\!\left(\sum_{i=1}^{2} \exp\!\left(-\frac{\|p - p_i^\text{exit}\|^2}{\tau}\right)\right)$$

Gradient: $\nabla C_\text{goal} = \sum_i w_i \nabla C_i$, where
$w_i = \exp(-C_i/\tau)\big/\sum_j \exp(-C_j/\tau)$.  Recommended $\tau \in [0.5, 1.5]$.

**Height:**

$$C_\text{height} = w_h\,(z - z_\text{surf}(x,y))^2$$

**Walls** (log-barrier):

$$\phi(d) = \log\!\left(\frac{R}{d + \varepsilon}\right), \quad d \le R$$

**Repulsion:**

$$C_\text{repulsion} = \sum_{j \neq i} \phi(\|p_i - p_j\|), \quad \phi(d) = \tfrac{1}{2}(R_s - d)^2 \text{ for } d \le R_s$$

**Smoothness:**

$$C_\text{smooth} = \|p^{(k+1)} - p^{(k)}\|^2$$

---

## 4. Implementation

In [ ]:
import numpy as np
import os, glob, subprocess
try:
    from IPython.display import HTML, display
except ImportError:
    HTML = None
    display = print
from base64 import b64encode

# ── Building dimensions ──────────────────────────────────────────────────────
H       = 4.0          # floor-to-floor height
FLOOR_Z = [0.0, H, 2*H]   # z of ground / first / second floors
W, D    = 12.0, 10.0  # floor width (x) and depth (y)
RAMP_R  = 1.1          # ramp half-width (capsule radius)

# ── Simulation parameters ─────────────────────────────────────────────────────
N_PARTICLES = 24
STEPS       = 1500
DT          = 0.035
ALPHA_SMOOTH = 0.25    # EMA for velocity

# ── Cost weights ──────────────────────────────────────────────────────────────
W_WALL    = 10.0
W_HEIGHT  = 65.0
W_REPULS  = 2.0        # softer repulsion so agents queue at ramps without deadlock
W_SMOOTH  = 0.4
TAU       = 5.0        # soft-min temperature (linear distances — meaningful at 10+ m)
WALL_R    = 0.45       # MUST be < 0.5 (= exit-gap half-width) so gap corners don't block
SOCIAL_R  = 0.80       # smaller radius allows tight queuing at ramp entries
MAX_GRAD  = 25.0       # gradient clipping

# ── Exit tolerance ────────────────────────────────────────────────────────────
# EXIT_TOL counts an agent as evacuated once within this distance of the
# target point just outside the south wall (EXIT1/EXIT2 are at y=-1).
EXIT_TOL  = 1.5

# Each floor has a list of wall segments as (a, b) pairs (2-D, in XY plane).
# Origin is at the SW corner of the building on each floor.
# Outer walls are shared; interior walls differ per floor.

def make_floor_walls():
    """
    Returns dict with keys 0, 1, 2 (floor index), each a list of
    (np.array([x0,y0]), np.array([x1,y1])) wall segments.

    Ground floor  – two exits (gaps in the south wall)
    First floor   – L-shaped interior partition
    Second floor  – central dividing wall with a corridor gap
    """
    walls = {}

    # ── Ground floor (floor 0) ────────────────────────────────────────────────
    # Outer boundary with two exit gaps in the south wall.
    # Gap 1 centred at x=2: open [1.2, 2.8]  (width 1.6)
    # Gap 2 centred at x=10: open [9.2, 10.8] (width 1.6)
    g0 = [
        # South wall (bottom), with two exit gaps
        ([ 0,  0], [ 1.2, 0]),
        ([ 2.8, 0], [ 9.2, 0]),
        ([10.8, 0], [W,    0]),
        # North wall (top)
        ([ 0,  D], [W,    D]),
        # West wall
        ([ 0,  0], [ 0,   D]),
        # East wall
        ([W,   0], [W,    D]),
        # Interior: horizontal partition at y=5, gap at x=[5,7]
        ([ 0,   5], [ 5,   5]),
        ([ 7,   5], [W,    5]),
        # Interior: vertical stub at x=6, from y=5 to y=8
        ([ 6,   5], [ 6,   8]),
    ]
    walls[0] = [(np.array(a, float), np.array(b, float)) for a, b in g0]

    # ── First floor (floor 1) ─────────────────────────────────────────────────
    g1 = [
        # Outer boundary (no exits)
        ([ 0,  0], [W,    0]),
        ([ 0,  D], [W,    D]),
        ([ 0,  0], [ 0,   D]),
        ([W,   0], [W,    D]),
        # Interior: L-shaped partition
        # Vertical wall x=3 from south to y=6 (blocks west room)
        ([ 3,  0], [ 3,   6]),
        # Horizontal wall y=6 with gap x=[4.4,6.6] for Ramp A (centreline x=5.5, r=1.1)
        ([ 3,  6], [ 4.4, 6]),
        ([ 6.6,6], [ 8,   6]),
        # Alcove north wall (east of x=8)
        ([ 8,  6], [ 8,   9]),
    ]
    walls[1] = [(np.array(a, float), np.array(b, float)) for a, b in g1]

    # ── Second floor (floor 2) ────────────────────────────────────────────────
    g2 = [
        # Outer boundary
        ([ 0,  0], [W,    0]),
        ([ 0,  D], [W,    D]),
        ([ 0,  0], [ 0,   D]),
        ([W,   0], [W,    D]),
        # Interior: central east-west wall with gap at x=[4.5, 6.5]
        ([ 0,  5], [ 4.5, 5]),
        ([ 6.5, 5], [W,   5]),
        # North room divider
        ([ 9,  5], [ 9,  D]),
    ]
    walls[2] = [(np.array(a, float), np.array(b, float)) for a, b in g2]

    return walls


FLOOR_WALLS = make_floor_walls()

# ── Exits (ground floor, 3-D positions) ───────────────────────────────────────
# Targets placed slightly OUTSIDE the south wall so the goal gradient
# pulls agents cleanly through the exit gaps.
EXIT1 = np.array([ 2.0, -1.0, 0.0])   # south-west exit  (outside building)
EXIT2 = np.array([10.0, -1.0, 0.0])   # south-east exit  (outside building)
EXITS = [EXIT1, EXIT2]
# Display positions for visualization markers (at the door threshold)
EXIT1_DISP = np.array([ 2.0, 0.0, 0.0])
EXIT2_DISP = np.array([10.0, 0.0, 0.0])

# Ramp B: first floor → ground floor (west side, near EXIT 1)
# Centreline from (x=5, y=1) on ground floor to (x=5, y=4.5) on first floor.
RAMP_B = dict(
    A     = np.array([5.0, 1.0]),   # xy on ground floor (z=z0)
    B     = np.array([5.0, 4.5]),   # xy on first floor  (z=z1)
    z0    = FLOOR_Z[0],
    z1    = FLOOR_Z[1],
    r     = RAMP_R,
    floor_below = 0,
    floor_above = 1,
)

# Ramp C: first floor → ground floor (east side, near EXIT 2)
# Gives floor-1 east-wing agents a second descent path so Ramp B doesn't jam.
RAMP_C = dict(
    A     = np.array([9.5, 1.0]),   # xy on ground floor (z=z0)
    B     = np.array([9.5, 4.5]),   # xy on first floor  (z=z1)
    z0    = FLOOR_Z[0],
    z1    = FLOOR_Z[1],
    r     = RAMP_R,
    floor_below = 0,
    floor_above = 1,
)

# Ramp A: second floor → first floor
# Centreline at x=5.5 passes through the gap x=[4.4,6.5] in the floor-2 y=5 wall.
# Lower end A is placed at y=6.5 — 0.5 m NORTH of the floor-1 y=6 wall so the
# ramp surface never intersects that wall visually or geometrically.
# The floor-1 gap at y=6 (x=[4.4,6.6]) lets agents walk south after descending.
RAMP_A = dict(
    A     = np.array([5.5, 6.5]),   # xy on first floor  (z=z0) — north of y=6 wall
    B     = np.array([5.5, 9.0]),   # xy on second floor (z=z1)
    z0    = FLOOR_Z[1],
    z1    = FLOOR_Z[2],
    r     = RAMP_R,
    floor_below = 1,
    floor_above = 2,
)

RAMPS = [RAMP_A, RAMP_B, RAMP_C]

def point_to_segment_dist_2d(p, a, b):
    """Minimum distance from 2-D point p to segment [a, b]."""
    v = b - a
    w = p - a
    vv = np.dot(v, v)
    if vv < 1e-12:
        return np.linalg.norm(p - a)
    t = np.clip(np.dot(w, v) / vv, 0.0, 1.0)
    return np.linalg.norm(p - (a + t * v))


def ramp_u(xy, ramp):
    """Normalised coordinate along ramp centreline (0 at A, 1 at B)."""
    A, B = ramp["A"], ramp["B"]
    v = B - A
    vv = np.dot(v, v)
    if vv < 1e-12:
        return 0.0
    t = np.dot(xy - A, v) / vv
    return float(np.clip(t, 0.0, 1.0))


def ramp_height(xy, ramp):
    """Height of ramp surface at 2-D position xy (inside footprint only)."""
    u = ramp_u(xy, ramp)
    return ramp["z0"] + (ramp["z1"] - ramp["z0"]) * u


def in_ramp_footprint(xy, ramp):
    """True if xy is inside the ramp capsule footprint."""
    return point_to_segment_dist_2d(xy, ramp["A"], ramp["B"]) <= ramp["r"]


def surface_height(xy, z_agent):
    """
    Height of the building surface at 2-D position xy for an agent whose
    current z-coordinate is z_agent.

    Two activation cases for a ramp:
      1. Agent is near the ramp surface (mid-descent): |z_agent - z_r| <= TOL
      2. Agent is at the UPPER-FLOOR ENTRY (top 40 % of ramp by u):
         z_agent ≈ z1  AND  u >= 0.60
         This lets floor-N agents step onto the ramp from above without
         being sucked down to a mid-ramp height from far away.

    Case 2 is the critical fix: it prevents floor-2 agents (z=8) from being
    snapped to an intermediate ramp height (e.g. z_r=4.8) just because they
    happen to cross the ramp's XY footprint at an unrelated y position.
    """
    RAMP_TOL = 0.80
    for ramp in RAMPS:
        if in_ramp_footprint(xy, ramp):
            z_r = ramp_height(xy, ramp)
            # Case 1 — near the ramp surface (mid-descent or lower-floor entry)
            if abs(z_agent - z_r) <= RAMP_TOL:
                return z_r
            # Case 2 — at upper-floor level, entering from the top portion of ramp
            u = ramp_u(xy, ramp)
            if z_agent >= ramp["z1"] - RAMP_TOL and u >= 0.60:
                return z_r
    # Flat floor — snap to nearest floor level
    return min(FLOOR_Z, key=lambda fz: abs(z_agent - fz))

def wall_penalty(d, R=1.0, eps=0.05):
    """Truncated log-barrier wall penalty."""
    if d <= R:
        return np.log(R / (d + eps))
    return 0.0


def goal_softmin(p3, exits, tau=TAU):
    """
    Soft-min over XY LINEAR distances to each exit.

    Using LINEAR (not squared) distances keeps exp(-d/tau) non-negligible even
    for upper-floor agents far from the exit. With tau=5, an agent 12 m away
    gets exp(-12/5)≈0.09 — enough gradient to pull it toward the exit.

    C_goal = -tau * log(sum_i exp(-||p_xy - exit_i_xy|| / tau))
    """
    p_xy = p3[:2]
    dists = np.array([np.linalg.norm(p_xy - e[:2]) for e in exits])
    # Numerically stable log-sum-exp with linear distances
    d_min = dists.min()
    lse = d_min - tau * np.log(np.sum(np.exp(-(dists - d_min) / tau)))
    return lse


def height_cost(p3, floor_z, w=W_HEIGHT):
    """Penalise deviation from the building surface."""
    xy = p3[:2]
    z_surf = surface_height(xy, floor_z)
    return w * (p3[2] - z_surf) ** 2


def agent_repulsion(pi, pj, R=SOCIAL_R):
    """Quadratic repulsion between two agents."""
    d = np.linalg.norm(pi - pj)
    if d < 1e-9 or d > R:
        return 0.0
    return 0.5 * (R - d) ** 2

# ── Ramp guidance cost ────────────────────────────────────────────────────────
W_RAMP_GUIDE = 14.0   # weight for ramp-guidance term
W_GAP_GUIDE  = 8.0    # weight for interior corridor-gap guidance

# Gap waypoints: each entry is (gap_centre_xy, sigma_sq, condition(p3)->bool)
# Floor 0: horizontal wall at y=5, gap at x=[5,7]     → pull toward x=6
# Floor 2: horizontal wall at y=5, gap at x=[4.5,6.5] → pull toward x=5.5
#   (same x as Ramp A, so guidance funnels directly toward ramp entry)
_GAP_WAYPOINTS = [
    # Floor 0 — north-room agents toward gap centre x=6, y=5
    (np.array([6.0, 5.0, 0.0]), (3.5)**2,
     lambda p: p[2] < 0.5 and p[1] > 5.0),
    # Floor 1 — agents north of y=6.3 (just descended Ramp A) toward gap centre x=5.5, y=6
    # This steers them into the wall gap so they can continue south to Ramp B/C
    (np.array([5.5, 6.0, 4.0]), (2.0)**2,
     lambda p: abs(p[2] - FLOOR_Z[1]) < 0.6 and p[1] > 6.3),
    # Floor 2 — south-room agents toward gap centre x=5.5, y=5 → then Ramp A
    (np.array([5.5, 5.0, 8.0]), (4.0)**2,
     lambda p: p[2] > 7.5 and p[1] < 5.5),
]

def gap_guidance_cost(p3):
    """
    Pulls agents toward interior wall gaps so they can navigate to their ramp.
    - Floor-0 north-room agents → gap at x=[5,7], y=5
    - Floor-2 south-room agents → gap at x=[4.5,6.5], y=5 (to reach Ramp A)
    """
    c = 0.0
    for centre, sig2, cond in _GAP_WAYPOINTS:
        if not cond(p3):
            continue
        dwall = abs(p3[1] - centre[1])
        w = np.exp(-(dwall**2) / sig2)
        dx = p3[0] - centre[0]
        c += W_GAP_GUIDE * w * dx**2
    return c

def ramp_guidance_cost(p3):
    """
    Pulls each agent toward the NEAREST ramp that serves its current floor.

    FIX: uses point_to_segment_dist_2d (correct capsule-boundary distance)
    instead of the old |xy - B|² - r² approximation, which incorrectly gave
    zero cost for agents near the B endpoint but NOT inside the footprint.

    A tight Gaussian in z (sigma = 0.3*H) ensures no cross-floor contamination.
    Cost is zero inside the capsule footprint — height cost takes over there.
    """
    z, xy = p3[2], p3[:2]
    sigma_sq = (0.30 * H) ** 2
    c = 0.0

    from collections import defaultdict
    by_floor = defaultdict(list)
    for ramp in RAMPS:
        by_floor[ramp["floor_above"]].append(ramp)

    for fl_idx, fl_ramps in by_floor.items():
        z_upper = FLOOR_Z[fl_idx]
        w = np.exp(-((z - z_upper) ** 2) / (2.0 * sigma_sq))
        if w < 0.02:
            continue
        # Only pull toward the nearest ramp on this floor
        best_d_outside = None
        for ramp in fl_ramps:
            d_to_seg = point_to_segment_dist_2d(xy, ramp["A"], ramp["B"])
            d_outside = max(0.0, d_to_seg - ramp["r"])
            if best_d_outside is None or d_outside < best_d_outside:
                best_d_outside = d_outside
        c += w * (best_d_outside ** 2)

    return W_RAMP_GUIDE * c

def nearest_floor(z):
    """Return the index of the floor closest to z."""
    dists = [abs(z - fz) for fz in FLOOR_Z]
    return int(np.argmin(dists))


def agent_floor_z(p3):
    """
    Return the z of the floor on which the agent is currently standing.
    We pick the nearest floor whose level is ≤ agent z + small tolerance,
    falling back to the nearest floor if the agent is on a ramp.
    """
    xy = p3[:2]
    # Check if agent is on a ramp — use ramp's lower-floor z as reference
    for ramp in RAMPS:
        if in_ramp_footprint(xy, ramp):
            # The surface height at this xy is the ramp height
            return ramp_height(xy, ramp)
    # Otherwise snap to nearest flat floor
    return FLOOR_Z[nearest_floor(p3[2])]

def total_cost(p3_try, i, positions, floor_z_i, walls_2d, exits,
               active=None, z_surf_ref=None):
    """
    Evaluate the total cost for agent i at trial position p3_try.
    active     : boolean array length N; if given, only active agents repel.
    z_surf_ref : surface height at the agent's CURRENT xy (not the trial xy).
                 Using the current xy prevents the height cost from creating
                 spurious XY gradients that oppose navigation.
    """
    c = 0.0

    # ── Goal (soft-min) ───────────────────────────────────────────────────────
    c += goal_softmin(p3_try, exits, tau=TAU)

    # ── Ramp guidance (pulls upper-floor agents toward correct ramp entry) ────
    c += ramp_guidance_cost(p3_try)

    # ── Gap guidance (pulls floor-0 north-room agents toward wall gap) ────────
    c += gap_guidance_cost(p3_try)

    # ── Wall repulsion — not applied inside ramp footprints (ramp is a corridor)
    xy_try = p3_try[:2]
    if not any(in_ramp_footprint(xy_try, r) for r in RAMPS):
        floor_idx = nearest_floor(p3_try[2])
        floor_walls = FLOOR_WALLS.get(floor_idx, [])
        for a, b in floor_walls:
            d = point_to_segment_dist_2d(xy_try, a, b)
            c += W_WALL * wall_penalty(d, R=WALL_R)

    # ── Height adherence — use z_surf at the CURRENT xy so only z is penalised
    # (avoids large spurious XY gradients that fight navigation on ramps)  ────
    z_ref = z_surf_ref if z_surf_ref is not None else surface_height(xy_try, p3_try[2])
    c += W_HEIGHT * (p3_try[2] - z_ref) ** 2

    # ── Agent repulsion (skip evacuated; zero inside ramp — prevents traffic jams) ─
    p_in_ramp = any(in_ramp_footprint(p3_try[:2], r) for r in RAMPS)
    if not p_in_ramp:
        for j, pj in enumerate(positions):
            if j == i:
                continue
            if active is not None and not active[j]:
                continue
            # Also skip if the other agent is on a ramp (moving queue)
            if any(in_ramp_footprint(pj[:2], r) for r in RAMPS):
                continue
            c += W_REPULS * agent_repulsion(p3_try, pj)

    # ── Smoothness (penalise large step from current position) ────────────────
    c += W_SMOOTH * np.sum((p3_try - positions[i]) ** 2)

    return c


def gradient_3d(i, positions, exits, reached, eps=1e-4):
    """
    Numerical gradient of total cost for agent i via central differences.

    z is treated as a CONSTRAINT (hard surface projection) rather than a
    free variable. Only x and y gradients are computed here; z is
    determined by projecting onto the building surface after each step.
    This eliminates z oscillation that would otherwise stall ramp descent.

    Returns (3,) gradient vector with grad_z=0, clipped to MAX_GRAD.
    """
    pi = positions[i]
    floor_z_i = FLOOR_Z[nearest_floor(pi[2])]
    walls_2d = []
    active = ~reached
    z_surf_ref = surface_height(pi[:2], pi[2])
    grad = np.zeros(3)

    for dim in range(2):   # XY only — z is constrained by projection
        pp = pi.copy(); pp[dim] += eps
        pm = pi.copy(); pm[dim] -= eps

        cp = total_cost(pp, i, positions, floor_z_i, walls_2d, exits, active,
                        z_surf_ref=z_surf_ref)
        cm = total_cost(pm, i, positions, floor_z_i, walls_2d, exits, active,
                        z_surf_ref=z_surf_ref)
        grad[dim] = (cp - cm) / (2.0 * eps)

    # grad[2] stays 0 — z is set by surface projection in the integrator

    g_norm = np.linalg.norm(grad[:2])
    if g_norm > MAX_GRAD:
        grad[:2] = grad[:2] * MAX_GRAD / g_norm

    return grad

def is_at_exit(p3, exits, tol=EXIT_TOL):
    """True if agent has reached any exit."""
    return any(np.linalg.norm(p3 - e) < tol for e in exits)


def initial_positions(n=N_PARTICLES, seed=42):
    """
    Distribute n agents roughly equally across the three floors.
    Positions are chosen to be safely inside the floor plan.
    """
    rng = np.random.default_rng(seed)
    positions = []
    per_floor = n // 3
    extra = n - 3 * per_floor

    # Safe spawn regions (xmin, xmax, ymin, ymax) per floor
    # Floor 0: west south zone — clear of Ramp B (x=[3.9,6.1]) and Ramp C (x=[8.4,10.6])
    # Floor 1: just south of the y=6 wall so agents head straight to Ramp B/C
    # Floor 2: west side x=[0.5,4.0] — WEST of Ramp A's XY footprint (x=[4.4,6.6]).
    #           This prevents agents from spawning inside the footprint and being
    #           snapped to a mid-ramp z on the very first simulation step.
    spawn_regions = {
        0: (0.5, 3.8, 0.5, 4.0),    # ground floor, west side clear of ramp footprints
        1: (3.5, 11.5, 5.1, 5.5),   # first floor, south of y=6 wall
        2: (0.5, 4.0, 5.5, 9.5),    # second floor, west side (east edge 4.0 < ramp x 4.4)
    }

    counts = [per_floor, per_floor, per_floor + extra]
    for fl, cnt in enumerate(counts):
        xmin, xmax, ymin, ymax = spawn_regions[fl]
        for _ in range(cnt):
            x = rng.uniform(xmin, xmax)
            y = rng.uniform(ymin, ymax)
            z = FLOOR_Z[fl]
            positions.append([x, y, z])

    return np.array(positions, dtype=float)


def simulate_evacuation(positions_init, exits, steps=STEPS, dt=DT):
    """
    Run the 3-D evacuation simulation with momentum (EMA velocity) smoothing.

    At each step the gradient gives a desired velocity.  Instead of applying
    it directly, we blend it with the previous step's velocity using an
    exponential moving average controlled by ALPHA_SMOOTH:

        velocity[i] ← ALPHA_SMOOTH * velocity[i]  +  (1 - ALPHA_SMOOTH) * (-dt * g)
        position[i] ← position[i] + velocity[i]

    This is equivalent to a low-pass filter on the direction of motion,
    suppressing rapid oscillations (jitter) near walls and ramp edges.
    ALPHA_SMOOTH = 0 → pure gradient descent (no memory).
    ALPHA_SMOOTH = 1 → agent keeps last velocity forever (no correction).
    A value of 0.25 gives gentle damping while staying responsive.

    Returns
    -------
    history : (steps+1, N, 3) position array
    reached : (N,) bool array — True if agent reached an exit
    """
    N = len(positions_init)
    positions = positions_init.copy()
    velocity  = np.zeros_like(positions)   # EMA velocity, initially zero
    history   = [positions.copy()]
    reached   = np.zeros(N, dtype=bool)

    for step in range(steps):
        new_pos = positions.copy()
        new_vel = velocity.copy()

        for i in range(N):
            if reached[i]:
                new_vel[i] = 0.0
                continue
            if is_at_exit(positions[i], exits):
                reached[i] = True
                new_vel[i] = 0.0
                continue

            g = gradient_3d(i, positions, exits, reached)

            # ── Momentum-smoothed velocity (EMA) ─────────────────────────
            desired_step = -dt * g
            new_vel[i] = (ALPHA_SMOOTH * velocity[i]
                          + (1.0 - ALPHA_SMOOTH) * desired_step)

            candidate = positions[i] + new_vel[i]

            # ── Hard-project z onto building surface ──────────────────────
            xy_new = candidate[:2]
            candidate[2] = surface_height(xy_new, candidate[2])

            # ── Keep agent inside building bounding box ───────────────────
            candidate[0] = np.clip(candidate[0], 0.05, W - 0.05)
            candidate[1] = np.clip(candidate[1], -1.5, D - 0.05)

            new_pos[i] = candidate

        positions = new_pos
        velocity  = new_vel
        history.append(positions.copy())

        if reached.all():
            while len(history) <= steps:
                history.append(positions.copy())
            break

    print(f"  Evacuated: {reached.sum()}/{N} agents "
          f"after {min(step+1, steps)} steps.")
    return np.array(history[:steps+1]), reached

print("Initialising agent positions …")
pos_init = initial_positions(N_PARTICLES)
print(f"  {N_PARTICLES} agents across 3 floors.")
print(f"  Floor 0 (ground): {(pos_init[:,2] == FLOOR_Z[0]).sum()} agents")
print(f"  Floor 1 (first):  {(pos_init[:,2] == FLOOR_Z[1]).sum()} agents")
print(f"  Floor 2 (second): {(pos_init[:,2] == FLOOR_Z[2]).sum()} agents")

print("\nRunning evacuation simulation …")
history, reached = simulate_evacuation(pos_init, EXITS)
print(f"History shape: {history.shape}  (frames × agents × 3)")


---

## 5. Trajectory Analysis (2-D projections per floor)

Before the 3-D vedo animation, we visualise the XY trajectories on each floor's
floor plan to verify that agents follow plausible paths and respect wall boundaries.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection

FLOOR_COLORS = ["#e74c3c", "#3498db", "#2ecc71"]
FLOOR_NAMES  = ["Ground floor (z=0)", f"First floor (z={H})", f"Second floor (z={2*H})"]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for fl, ax in enumerate(axes):
    fz = FLOOR_Z[fl]
    ax.set_xlim(-0.5, W + 0.5)
    ax.set_ylim(-0.5, D + 0.5)
    ax.set_aspect("equal")
    ax.set_title(FLOOR_NAMES[fl], fontsize=11)
    ax.set_xlabel("x"); ax.set_ylabel("y")
    ax.grid(True, alpha=0.25)

    # Draw floor walls
    for a, b in FLOOR_WALLS[fl]:
        ax.plot([a[0], b[0]], [a[1], b[1]], "k-", lw=2)

    # Draw ramp footprints
    for ramp in RAMPS:
        if ramp["floor_below"] == fl or ramp["floor_above"] == fl:
            A2, B2 = ramp["A"], ramp["B"]
            ax.plot([A2[0], B2[0]], [A2[1], B2[1]], "m-", lw=4, alpha=0.4,
                    label="Ramp centreline")
            # Draw capsule outline (approximate with a thick line)
            ax.plot([A2[0], B2[0]], [A2[1], B2[1]], "m-",
                    lw=ramp["r"] * 60, alpha=0.12, solid_capstyle="round")

    # Draw exits on ground floor
    if fl == 0:
        for k, ex in enumerate(EXITS):
            ax.plot(ex[0], ex[1], "r*", ms=16, zorder=8,
                    label=f"Exit {k+1}")
        ax.legend(fontsize=8, loc="upper right")

    # Draw agent trajectories for agents that started on this floor
    for i in range(N_PARTICLES):
        # Find steps where agent is roughly on this floor
        traj = history[:, i, :]
        on_floor = np.abs(traj[:, 2] - fz) < (H * 0.6)
        if not on_floor.any():
            continue
        xs = traj[on_floor, 0]
        ys = traj[on_floor, 1]
        ax.plot(xs, ys, "-", color=FLOOR_COLORS[fl], alpha=0.5, lw=1.2)
        ax.scatter(xs[0], ys[0], s=40, color=FLOOR_COLORS[fl],
                   edgecolors="black", lw=0.5, zorder=6)

plt.suptitle("Agent Trajectories per Floor (XY Projection)", fontsize=13)
plt.tight_layout()
plt.savefig("/content/trajectories_per_floor.png", dpi=120,
            bbox_inches="tight")
plt.show()
print("Trajectory plot saved.")


---

## 6. Exit Selection Analysis

We measure how many agents chose each exit to verify that the soft-min cost
function allows natural splitting without any hard-coded assignment.

In [ ]:
def assigned_exit(final_pos, exits, tol=2.5):
    """Return 0, 1, or -1 (not evacuated) for final position."""
    for k, e in enumerate(exits):
        if np.linalg.norm(final_pos - e) < tol:
            return k
    return -1

final_positions = history[-1]
exit_counts = [0, 0, 0]   # exit0, exit1, not evacuated
for i in range(N_PARTICLES):
    k = assigned_exit(final_positions[i], EXITS)
    if k >= 0:
        exit_counts[k] += 1
    else:
        exit_counts[2] += 1

print("\nExit selection summary:")
print(f"  Exit 1 (SW): {exit_counts[0]} agents")
print(f"  Exit 2 (SE): {exit_counts[1]} agents")
print(f"  Not evacuated (still inside): {exit_counts[2]} agents")

fig2, ax2 = plt.subplots(figsize=(5, 4))
bars = ax2.bar(["Exit 1 (SW)", "Exit 2 (SE)", "Not evacuated"],
               exit_counts, color=["#e74c3c", "#3498db", "#95a5a6"])
ax2.set_ylabel("Number of agents")
ax2.set_title("Exit Selection — Soft-Min Goal Cost (τ = {})".format(TAU))
for bar, val in zip(bars, exit_counts):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
             str(val), ha="center", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig("/content/exit_selection.png", dpi=120,
            bbox_inches="tight")
plt.show()
print("Exit selection chart saved.")

---

## 7. 3-D Architectural Visualization and Animation

The building is rendered with matplotlib's Poly3DCollection using a
**Lambertian (diffuse) shading** model:

    I = ambient + (1 - ambient) * max(0, n_hat · l_hat)

where l_hat is a fixed directional light and n_hat is each polygon's
outward normal.  This matches the style of the assignment figure:

  * White background
  * Solid-colour walls rendered as 3-D boxes (thickness clearly visible)
  * TALL back (north) walls to frame the scene — no transparency needed
  * LOW interior walls so agent spheres are always visible above them
  * Ramp surface patches as inclined parallelogram meshes
  * Drop shadows under each sphere on the floor below them

In [ ]:
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from matplotlib import animation as manim

# ── Lambertian lighting ───────────────────────────────────────────────────────
_lv = np.array([0.6, -0.9, 1.8])
LIGHT_DIR = _lv / np.linalg.norm(_lv)
AMBIENT   = 0.30

def _shade(normal, rgb, alpha=1.0):
    """RGBA colour via Lambertian diffuse model."""
    n = np.asarray(normal, float)
    nm = np.linalg.norm(n)
    if nm > 1e-9: n /= nm
    I = AMBIENT + (1.0 - AMBIENT) * max(0.0, float(np.dot(n, LIGHT_DIR)))
    return (min(1., rgb[0]*I), min(1., rgb[1]*I), min(1., rgb[2]*I), alpha)

# ── Wall heights by type ──────────────────────────────────────────────────────
# Exterior back/side walls = full floor height (H) so the building is framed.
# Interior walls intentionally SHORT so agent spheres are always visible above.
# Front walls near-zero to preserve the open "cutaway" view from the camera.
_W_H_MAP = {"back": H, "side": H * 0.85, "front": 0.30, "interior": 0.75}
WALL_THICK = 0.32

def _wall_type(a2, b2):
    mx = (a2[0] + b2[0]) / 2.0; my = (a2[1] + b2[1]) / 2.0
    if abs(my - D) < 0.4: return "back"
    if abs(my)     < 0.4: return "front"
    if abs(mx)     < 0.4 or abs(mx - W) < 0.4: return "side"
    return "interior"

# ── Polygon builders ──────────────────────────────────────────────────────────
def _box_polys(a2, b2, z_floor, height, thick=WALL_THICK,
               rgb=(0.90, 0.88, 0.85)):
    """6 Lambertian-shaded quad faces for one wall box."""
    d = b2 - a2; L = np.linalg.norm(d)
    if L < 1e-6: return []
    dh = d / L; perp = np.array([-dh[1], dh[0]]); t = thick / 2.0
    z0, z1 = z_floor, z_floor + height
    def v(xy, z): return np.array([xy[0], xy[1], z])
    c = [v(a2+perp*t, z0), v(a2-perp*t, z0), v(b2-perp*t, z0), v(b2+perp*t, z0),
         v(a2+perp*t, z1), v(a2-perp*t, z1), v(b2-perp*t, z1), v(b2+perp*t, z1)]
    faces = [
        ([c[4], c[5], c[6], c[7]], (0,        0,        1)),
        ([c[0], c[3], c[7], c[4]], ( perp[0],  perp[1], 0)),
        ([c[1], c[2], c[6], c[5]], (-perp[0], -perp[1], 0)),
        ([c[0], c[1], c[5], c[4]], (-dh[0],   -dh[1],   0)),
        ([c[3], c[2], c[6], c[7]], ( dh[0],    dh[1],   0)),
    ]
    return [(np.array(vt), _shade(n, rgb)) for vt, n in faces]

SLAB_H = 0.20
_FLOOR_RGB = [(0.95, 0.92, 0.87), (0.89, 0.93, 0.97), (0.91, 0.96, 0.90)]
# Alpha for the floor TOP surface.  Semi-transparent so walls, ramps and
# agents on lower floors remain visible through the slab of the floor above.
# Ground floor (fl=0) is fully opaque — nothing is below it.
_SLAB_TOP_ALPHA = [0.22, 0.22, 0.22]  # all floors semi-transparent so agents are visible

def _slab_polys(z_floor, fl_idx):
    """Floor slab: semi-transparent TOP face + 4 opaque side faces.

    Upper-floor tops use alpha=0.28 so the walls, ramps and agent spheres
    on lower floors are clearly visible through the slab above them.
    The side (edge) faces stay fully opaque to show the slab thickness.
    """
    rgb   = _FLOOR_RGB[fl_idx]
    alpha = _SLAB_TOP_ALPHA[fl_idx]
    z0, z1 = z_floor - SLAB_H, z_floor
    c = np.array([[0,0,z0],[W,0,z0],[W,D,z0],[0,D,z0],
                  [0,0,z1],[W,0,z1],[W,D,z1],[0,D,z1]])
    top_face  = ([c[4],c[5],c[6],c[7]], (0,  0,  1))
    edge_faces = [
        ([c[0],c[1],c[5],c[4]], (0, -1,  0)),
        ([c[1],c[2],c[6],c[5]], (1,  0,  0)),
        ([c[2],c[3],c[7],c[6]], (0,  1,  0)),
        ([c[3],c[0],c[4],c[7]], (-1, 0,  0)),
    ]
    polys  = [(np.array(top_face[0]),  _shade(top_face[1],  rgb, alpha))]
    polys += [(np.array(vt),           _shade(n,            rgb, 1.0))
              for vt, n in edge_faces]
    return polys

_RAMP_RGB  = (0.76, 0.64, 0.46)
_RAMP_SRGB = (0.60, 0.50, 0.36)

def _ramp_polys(ramp):
    """Top surface + two side edges of a ramp, Lambertian-shaded."""
    A, B = ramp["A"], ramp["B"]; z0, z1, r = ramp["z0"], ramp["z1"], ramp["r"]
    v = B - A; vh = v / np.linalg.norm(v); perp = np.array([-vh[1], vh[0]])
    def pt(xy, z): return np.array([xy[0], xy[1], z])
    tl=pt(A+perp*r,z0); tr=pt(A-perp*r,z0); br=pt(B-perp*r,z1); bl=pt(B+perp*r,z1)
    e1=tr-tl; e2=bl-tl; norm=np.cross(e1,e2); norm/=max(np.linalg.norm(norm),1e-9)
    polys = [(np.array([tl,tr,br,bl]), _shade(norm, _RAMP_RGB, 0.96))]
    bbl=tl.copy(); bbl[2]-=0.14; bbr=tr.copy(); bbr[2]-=0.14
    ebl=bl.copy(); ebl[2]-=0.14; ebr=br.copy(); ebr[2]-=0.14
    polys += [
        (np.array([tl,bl,ebl,bbl]), _shade(( perp[0], perp[1],0), _RAMP_SRGB, 0.92)),
        (np.array([tr,br,ebr,bbr]), _shade((-perp[0],-perp[1],0), _RAMP_SRGB, 0.92)),
    ]
    return polys

def _exit_disc_polys(cx, cy, rgb, n_seg=22, r=0.55):
    """Flat coloured disc at exit location."""
    polys = []
    for s in range(n_seg):
        a1 = 2*np.pi*s/n_seg; a2_ = 2*np.pi*(s+1)/n_seg
        tri = np.array([[cx, cy, 0.025],
                        [cx+r*np.cos(a1),  cy+r*np.sin(a1),  0.025],
                        [cx+r*np.cos(a2_), cy+r*np.sin(a2_), 0.025]])
        polys.append((tri, _shade((0,0,1), rgb, 0.95)))
    return polys

# ── Pre-build static scene ────────────────────────────────────────────────────
print("\nBuilding 3-D architectural scene …")
_static_polys = []
for fi, fz in enumerate(FLOOR_Z):
    _static_polys.extend(_slab_polys(fz, fi))
for fl, fz in enumerate(FLOOR_Z):
    for a, b in FLOOR_WALLS[fl]:
        wt = _wall_type(a, b)
        _static_polys.extend(_box_polys(a, b, fz, _W_H_MAP[wt]))
for ramp in RAMPS:
    _static_polys.extend(_ramp_polys(ramp))
_static_polys.extend(_exit_disc_polys(EXIT1_DISP[0], EXIT1_DISP[1], (0.90, 0.20, 0.15)))
_static_polys.extend(_exit_disc_polys(EXIT2_DISP[0], EXIT2_DISP[1], (0.90, 0.52, 0.10)))

# Sort back-to-front by mean z (painter's algorithm)
_static_polys.sort(key=lambda pf: np.mean(pf[0][:, 2]))

# Agent colours (by originating floor)
_AG_COLORS = ["#e74c3c", "#3498db", "#2ecc71"]
_origin_fl  = [nearest_floor(pos_init[i, 2]) for i in range(N_PARTICLES)]
_agent_hex  = [_AG_COLORS[f] for f in _origin_fl]

def _setup_ax(fig, ax):
    ax.set_xlim(-1, W+1); ax.set_ylim(-2.0, D+0.5); ax.set_zlim(-0.3, 2*H+3.0)
    ax.set_box_aspect([W+2, D+2.5, 2*H+3.3])
    ax.set_axis_off(); ax.set_facecolor("white"); fig.patch.set_facecolor("white")
    ax.view_init(elev=42, azim=225)   # high elevation → all 3 floors visible

def _hex_to_rgb(h):
    """Convert '#rrggbb' to (r,g,b) floats."""
    return (int(h[1:3],16)/255, int(h[3:5],16)/255, int(h[5:7],16)/255)

# ── Agent sphere parameters ───────────────────────────────────────────────────
_SPH_R   = 0.32    # sphere radius
_SPH_LAT = 6       # latitude bands  (more = smoother, slower)
_SPH_LON = 10      # longitude slices
_SPH_ELEV = _SPH_R # sphere centre lifted one radius above surface

def _sphere_polys(cx, cy, cz_centre, R, rgb):
    """
    Approximate a sphere with (_SPH_LAT × _SPH_LON) quad patches.

    Each quad is shaded with Lambertian lighting using its OUTWARD NORMAL
    (cos θ cos φ, cos θ sin φ, sin θ).  This produces the characteristic
    bright-top / dark-side gradient that makes the agents look like spheres
    and explicitly implements the Lambertian (diffuse) model the instructor
    specified.
    """
    polys = []
    thetas = np.linspace(-np.pi/2, np.pi/2, _SPH_LAT + 1)
    phis   = np.linspace(0, 2*np.pi, _SPH_LON, endpoint=False)

    for i in range(_SPH_LAT):
        th0, th1 = thetas[i], thetas[i + 1]
        th_mid   = (th0 + th1) / 2.0
        r0 = R * np.cos(th0);  r1 = R * np.cos(th1)
        z0 = cz_centre + R * np.sin(th0)
        z1 = cz_centre + R * np.sin(th1)

        for j in range(_SPH_LON):
            ph0   = phis[j]
            ph1   = phis[(j + 1) % _SPH_LON]
            ph_mid = (ph0 + ph1) / 2.0

            v00 = [cx + r0*np.cos(ph0), cy + r0*np.sin(ph0), z0]
            v01 = [cx + r0*np.cos(ph1), cy + r0*np.sin(ph1), z0]
            v11 = [cx + r1*np.cos(ph1), cy + r1*np.sin(ph1), z1]
            v10 = [cx + r1*np.cos(ph0), cy + r1*np.sin(ph0), z1]

            # Outward normal at patch centroid on unit sphere
            normal = (np.cos(th_mid)*np.cos(ph_mid),
                      np.cos(th_mid)*np.sin(ph_mid),
                      np.sin(th_mid))

            verts = np.array([v00, v01, v11, v10])
            polys.append((verts, _shade(normal, rgb)))

    return polys


def _cast_shadow_poly(cx, cy, fz, R=_SPH_R):
    """
    Directional elliptical cast shadow projected onto the floor slab.

    The shadow is offset in the direction OPPOSITE to the horizontal
    component of LIGHT_DIR, and stretched along that axis — matching how
    a point light source above-and-behind would cast a shadow.
    """
    # Horizontal light projection → shadow falls the opposite way
    lx, ly = LIGHT_DIR[0], LIGHT_DIR[1]
    light_angle = np.arctan2(ly, lx)
    shadow_angle = light_angle + np.pi          # opposite to light
    cos_a, sin_a = np.cos(shadow_angle), np.sin(shadow_angle)

    major = R * 0.65    # elongated along shadow direction
    minor = R * 0.35    # compressed perpendicularly

    n = 14
    t = np.linspace(0, 2*np.pi, n, endpoint=False)
    # Offset centre away from light source
    cx_s = cx + cos_a * R * 0.45
    cy_s = cy + sin_a * R * 0.45

    ex = major * np.cos(t)
    ey = minor * np.sin(t)
    rx = cx_s + cos_a * ex - sin_a * ey
    ry = cy_s + sin_a * ex + cos_a * ey
    verts = np.column_stack([rx, ry, np.full(n, fz + 0.015)])
    return (verts, (0.08, 0.08, 0.08, 0.50))


def _agent_frame_polys(pos_f):
    """
    Build per-frame agent geometry as depth-sortable Poly3DCollection polygons.

    Each agent is:
      1. An elliptical cast shadow on the floor (directional, offset from light)
      2. A Lambertian-shaded sphere mesh (latitude × longitude quad patches)

    All polygons use the same (verts, rgba) format as _static_polys so they
    can be merged into a single z-sorted draw list with the architecture.
    """
    polys = []
    for i in range(N_PARTICLES):
        xi, yi, zi = pos_f[i]
        fz = FLOOR_Z[nearest_floor(zi)]

        # 1. Cast shadow (directional ellipse on floor)
        polys.append(_cast_shadow_poly(xi, yi, fz))

        # 2. Sphere mesh (Lambertian per-patch shading)
        rgb = _hex_to_rgb(_agent_hex[i])
        cz  = zi + _SPH_ELEV
        polys.extend(_sphere_polys(xi, yi, cz, _SPH_R, rgb))

    return polys


def _draw_frame(ax, frame_idx):
    """
    Render one 3-D frame.

    Agents are flat disc polygons merged into the same Poly3DCollection
    draw list as the walls and sorted by z-centroid (painter's algorithm).
    This is the ONLY approach that correctly depth-sorts agents on upper
    floors above lower-floor wall geometry in matplotlib 3-D.
    """
    ax.cla()
    _setup_ax(ax.figure, ax)

    pos_f     = history[frame_idx]
    frame_polys = _static_polys + _agent_frame_polys(pos_f)

    # Sort all polygons back-to-front by mean z of their vertices
    frame_polys.sort(key=lambda pf: np.mean(pf[0][:, 2]))

    # Single batched Poly3DCollection draw — one collection per polygon
    # keeps individual z-sort intact (batching all into one would ignore sort)
    for verts, fc in frame_polys:
        ax.add_collection3d(
            Poly3DCollection([verts], facecolors=[fc],
                              edgecolors="none", linewidth=0))

# ── Render MP4 ────────────────────────────────────────────────────────────────
def render_video(history, out_dir="frames_evac3d",
                 video_name="evacuation_3d.mp4", fps=20):
    """Save MP4 via ffmpeg AND return a FuncAnimation for inline display."""
    os.makedirs(out_dir, exist_ok=True)
    for f in glob.glob(os.path.join(out_dir, "*.png")): os.remove(f)
    T = history.shape[0]
    indices = np.linspace(0, T-1, min(150, T), dtype=int)
    print(f"  Rendering {len(indices)} frames for MP4 …")
    from matplotlib.lines import Line2D
    leg_h = [Line2D([0],[0], marker='o', color='w',
                    markerfacecolor=_AG_COLORS[f], markersize=9,
                    label=FLOOR_NAMES[f]) for f in range(3)]
    fig = plt.figure(figsize=(12, 9), dpi=110)
    ax  = fig.add_subplot(111, projection="3d")
    for k, fi in enumerate(indices):
        _draw_frame(ax, fi)
        ax.legend(handles=leg_h, loc="upper left", fontsize=9, framealpha=0.90)
        ax.set_title(f"Building Evacuation — step {fi}", fontsize=11, pad=4)
        fig.savefig(os.path.join(out_dir, f"frame_{k:05d}.png"),
                    bbox_inches="tight", facecolor="white", dpi=110)
        if k % 50 == 0: print(f"    frame {k}/{len(indices)}")
    plt.close(fig)
    print(f"  Rendered {len(indices)} frames.")
    # even-dimension filter prevents ffmpeg width/height error
    mp4_path = f"/content/{video_name}"
    cmd = (f"ffmpeg -loglevel error -y -framerate {fps} "
           f"-i '{out_dir}/frame_%05d.png' "
           f"-vf 'scale=trunc(iw/2)*2:trunc(ih/2)*2' "
           f"-c:v libx264 -pix_fmt yuv420p -movflags +faststart "
           f"'{mp4_path}'")
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if res.returncode == 0:
        print(f"  MP4 saved → {mp4_path}")
    else:
        print("  ffmpeg error:", res.stderr[:300])

print("Rendering 3-D evacuation animation …")
render_video(history)

# ── Inline 3-D animation for Colab ────────────────────────────────────────────
# Uses _draw_frame (disc-based depth-sorted Poly3DCollection) so all agents
# on all three floors are always visible. Kept to ~80 frames so jshtml
# generation completes in under a minute.
print("\nBuilding inline 3-D jshtml animation …")

from matplotlib import animation as _manim3
from matplotlib.lines import Line2D as _L3D

_anim3_step = max(1, history.shape[0] // 50)
_anim3_idx  = list(range(0, history.shape[0], _anim3_step))

_fig3d_ani = plt.figure(figsize=(11, 8))
_ax3d_ani  = _fig3d_ani.add_subplot(111, projection="3d")

_leg3d = [_L3D([0],[0], marker='o', color='w',
               markerfacecolor=_AG_COLORS[f], markersize=9,
               label=FLOOR_NAMES[f]) for f in range(3)]

def _update_3d(k):
    fi = _anim3_idx[k]
    _draw_frame(_ax3d_ani, fi)
    _ax3d_ani.legend(handles=_leg3d, loc="upper left", fontsize=9, framealpha=0.90)
    _ax3d_ani.set_title(f"Building Evacuation — step {fi}", fontsize=11)
    return []

_ani3d = _manim3.FuncAnimation(
    _fig3d_ani, _update_3d, frames=len(_anim3_idx),
    interval=80, blit=False)

plt.rcParams["animation.html"] = "jshtml"
plt.close(_fig3d_ani)

if HTML is not None:
    display(HTML(_ani3d.to_jshtml()))
else:
    print("Run: from IPython.display import HTML; HTML(_ani3d.to_jshtml())")
# Uses 3 side-by-side floor-plan subplots (one per floor).
# Scatter objects are updated per frame — no full redraws — so this renders
# quickly AND guarantees all agents on all floors are always visible,
# bypassing matplotlib 3-D depth-sorting artefacts entirely.
print("\nBuilding inline 2-D floor-plan animation …")

from matplotlib import animation as manim
from matplotlib.lines import Line2D as _L2D

_anim_step = max(1, history.shape[0] // 50)
_anim_idx  = list(range(0, history.shape[0], _anim_step))

_fig2d, _ax2d = plt.subplots(1, 3, figsize=(17, 6), facecolor="white")

# ── Draw static floor plan on each subplot ────────────────────────────────────
for fl, ax in enumerate(_ax2d):
    ax.set_xlim(-0.5, W + 0.5); ax.set_ylim(-1.5, D + 0.5)
    ax.set_aspect("equal"); ax.set_facecolor("#f8f8f8")
    ax.set_title(FLOOR_NAMES[fl], fontsize=10, pad=4)
    ax.set_xlabel("x (m)"); ax.grid(True, alpha=0.20)
    if fl == 0: ax.set_ylabel("y (m)")

    ax.fill([0, W, W, 0], [0, 0, D, D], color="#ececec", zorder=0)

    for a, b in FLOOR_WALLS[fl]:
        ax.plot([a[0], b[0]], [a[1], b[1]], "k-", lw=2.0, zorder=3)

    for ramp in RAMPS:
        if ramp["floor_below"] == fl or ramp["floor_above"] == fl:
            A2, B2 = ramp["A"], ramp["B"]
            ax.plot([A2[0], B2[0]], [A2[1], B2[1]], color="#9b59b6",
                    lw=4, alpha=0.45, solid_capstyle="round", zorder=2)
            ax.plot([A2[0], B2[0]], [A2[1], B2[1]], color="#9b59b6",
                    lw=ramp["r"] * 68, alpha=0.10,
                    solid_capstyle="round", zorder=1)

    if fl == 0:
        for k_ex, ex in enumerate(EXITS):
            ax.plot(ex[0], ex[1], "r*", ms=16, zorder=9,
                    label=f"Exit {k_ex+1}")
        ax.legend(fontsize=8, loc="upper right")

# One scatter object per floor (updated each frame)
_sc2d = []
for fl, ax in enumerate(_ax2d):
    sc = ax.scatter([], [], s=90, zorder=10,
                    edgecolors="white", linewidths=0.6, alpha=0.95)
    _sc2d.append(sc)

_title2d = _fig2d.suptitle("Building Evacuation — step 0", fontsize=12, y=1.01)
plt.tight_layout()

def _update_2d(k):
    fi = _anim_idx[k]
    pos_f = history[fi]
    for fl in range(3):
        fz = FLOOR_Z[fl]
        mask = np.array([abs(pos_f[i, 2] - fz) < H * 0.6
                         for i in range(N_PARTICLES)])
        xs = pos_f[mask, 0]; ys = pos_f[mask, 1]
        colors = [_agent_hex[i] for i in range(N_PARTICLES) if mask[i]]
        _sc2d[fl].set_offsets(
            np.c_[xs, ys] if len(xs) > 0 else np.zeros((0, 2)))
        if colors:
            _sc2d[fl].set_facecolors(colors)
    _title2d.set_text(f"Building Evacuation — step {fi}")
    return _sc2d

_ani2d = manim.FuncAnimation(
    _fig2d, _update_2d, frames=len(_anim_idx),
    interval=60, blit=False)

plt.rcParams["animation.html"] = "jshtml"
plt.close(_fig2d)

# Display inline in Colab
if HTML is not None:
    display(HTML(_ani2d.to_jshtml()))
else:
    print("Run: from IPython.display import HTML; HTML(_ani2d.to_jshtml())")

# ── Static snapshots ──────────────────────────────────────────────────────────
print("Saving static snapshots …")
from matplotlib.lines import Line2D
leg_h = [Line2D([0],[0], marker='o', color='w',
                markerfacecolor=_AG_COLORS[f], markersize=10,
                label=FLOOR_NAMES[f]) for f in range(3)]
for snap, fi in [("initial", 0),
                 ("mid",     history.shape[0] // 2),
                 ("final",   history.shape[0] - 1)]:
    fig_s, ax_s = plt.subplots(figsize=(12, 9), subplot_kw={"projection": "3d"})
    _draw_frame(ax_s, fi)
    ax_s.legend(handles=leg_h, loc="upper left", fontsize=9, framealpha=0.90)
    ax_s.set_title(f"Building Evacuation — {snap}", fontsize=12, pad=5)
    fig_s.savefig(f"/content/snapshot_{snap}.png",
                  bbox_inches="tight", facecolor="white", dpi=150)
    plt.close(fig_s)
    print(f"  Saved snapshot_{snap}.png")



Building 3-D architectural scene …
Rendering 3-D evacuation animation …
  Rendering 150 frames for MP4 …
    frame 0/150
    frame 50/150


---

## 8. Quantitative Analysis

### 8.1 Distance to Nearest Exit Over Time

In [ ]:
def dist_to_nearest_exit(history, exits):
    """(T, N) array — each agent's distance to its closest exit at each step."""
    T, N, _ = history.shape
    out = np.zeros((T, N))
    for t in range(T):
        for i in range(N):
            out[t, i] = min(np.linalg.norm(history[t, i] - e) for e in exits)
    return out

dists = dist_to_nearest_exit(history, EXITS)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: mean distance to exit over time
ax = axes[0]
T_hist = history.shape[0]
t_arr = np.arange(T_hist) * DT
ax.plot(t_arr, dists.mean(axis=1), color="#2c3e50", lw=2)
ax.fill_between(t_arr, dists.min(axis=1), dists.max(axis=1),
                alpha=0.2, color="#2c3e50")
ax.set_xlabel("Time (s)"); ax.set_ylabel("Distance to nearest exit")
ax.set_title("Mean (±range) Distance to Exit")
ax.grid(alpha=0.3)

# Panel 2: cumulative evacuees
ax = axes[1]
evacuated_per_step = np.sum(dists < EXIT_TOL, axis=1)
ax.step(t_arr, evacuated_per_step, color="#27ae60", lw=2)
ax.set_xlabel("Time (s)"); ax.set_ylabel("Cumulative evacuees")
ax.set_title(f"Cumulative Evacuees (N={N_PARTICLES})")
ax.set_ylim(0, N_PARTICLES + 1)
ax.grid(alpha=0.3)

plt.suptitle("Evacuation Performance Metrics", fontsize=13)
plt.tight_layout()
plt.savefig("/content/evacuation_metrics.png", dpi=120,
            bbox_inches="tight")
plt.show()
print("Metrics plot saved.")

---

## 9. Discussion

### Q1: How does the soft-min cost function enable natural exit selection?

The soft-min goal cost
$C_\text{goal}(p) = -\tau \log\!\bigl(\sum_i \exp(-\|p-p_i^\text{exit}\|^2 / \tau)\bigr)$
assigns each exit a continuous weight $w_i = \exp(-C_i/\tau) / \sum_j \exp(-C_j/\tau)$.
An agent is pulled toward whichever exit is currently closer, but the pull is smooth
everywhere — there is no discontinuity as in a hard minimum.  With $\tau = 1.0$
(moderately soft), agents near the midline split roughly 50/50 while agents clearly
closer to one exit are pulled decisively toward it.

### Q2: How do the ramps enable floor-to-floor transitions without discrete state?

The surface height $z_\text{surf}(x,y)$ transitions linearly along the ramp centreline.
The height-adherence cost $C_\text{height} = w_h(z - z_\text{surf})^2$ acts as a soft
spring pulling the agent onto the surface.  When an agent moves into the ramp footprint,
the target surface height smoothly changes from the flat floor to the inclined ramp
plane.  The gradient of $C_\text{height}$ in z drives the agent up or down accordingly.
There is no floor-state variable; the floor an agent "is on" is entirely encoded in its
continuous z coordinate.

### Q3: What role does the smoothness term play?

The smoothness penalty $C_\text{smooth} = \|p^{k+1} - p^k\|^2$ discourages large
steps.  Near the ramp transition — where the surface height gradient in z is steep —
without smoothing, an agent might overshoot vertically and oscillate.  The smoothness
term effectively acts as a momentum damper, keeping the trajectory smooth.

### Q4: Parameter sensitivity

| Parameter | Effect if too small | Effect if too large |
|-----------|--------------------|--------------------|
| $w_h$ (height weight) | Agents drift between floors | Gradient becomes stiff; slow convergence |
| $\tau$ (soft-min temp.) | Approaches hard-min; sharp boundaries | Both exits weighted equally; slow splitting |
| $w_\text{wall}$ | Agents clip through walls | Agents cannot navigate narrow corridors |
| $R_s$ (repulsion radius) | Agents overlap | Corridor throughput drops; jamming |



---

## 10. References

1. Helbing, D., & Molnár, P. (1995). Social force model for pedestrian dynamics. *Physical Review E*, 51(5), 4282–4286.
2. vedo — A python module for scientific analysis of 3D data. https://vedo.embl.es/
3. Ribeiro, E. (2026). CSE5280 Assignment: Evacuation Simulation (Three-Floor Building). Florida Institute of Technology.